In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2
import os

NPY_RESULTS_DIR = '/home/sdesai/THESIS/TFS-Diff-main/experiments/CT_Dose_SuperResolution_FinalRun_250923_232157/experiments/CT_Dose_SuperResolution_FinalRun/results/val_E19000_I0038000/npy/' 

SAMPLE_TAGS = [
    'b1_i0', 
    'b2_i0', 
    'b3_i0'
]

# This is the core function for the frequency analysis.
def plot_power_spectrum(image_np, title):
    """Calculates and plots the log power spectrum of a 2D numpy array."""
    # This converts the image from the spatial domain to the frequency domain.
    f_transform = np.fft.fft2(image_np)
    
    # By default, the low-frequency components are at the corners.
    # shift them to the center to make the spectrum easier to interpret.
    f_transform_shifted = np.fft.fftshift(f_transform)
    
    # Next, calculate the power spectrum, which is just the magnitude squared.
    power_spectrum = np.abs(f_transform_shifted)**2
    
    # The power spectrum has a huge dynamic range, hence a log scale
    # to make the details of both high and low frequencies visible.
    log_power_spectrum = np.log(power_spectrum + 1e-9) # Add a tiny value to avoid log(0)
    
    plt.imshow(log_power_spectrum, cmap='viridis')
    plt.title(title)
    plt.colorbar()
    plt.axis('off')

if __name__ == "__main__":
    # A quick check to make sure the results directory actually exists.
    if not os.path.isdir(NPY_RESULTS_DIR):
        print(f"Error: The directory '{NPY_RESULTS_DIR}' does not exist.")
        print("Please update the NPY_RESULTS_DIR variable in the script.")
    else:
        for tag in SAMPLE_TAGS:
            print(f"Processing sample: {tag}...")
            
            # Define the paths for the Low-Res, Super-Res, and High-Res (Ground Truth) images.
            path_lr = os.path.join(NPY_RESULTS_DIR, f'LR_UP_{tag}.npy')
            path_sr = os.path.join(NPY_RESULTS_DIR, f'SR_{tag}.npy')
            path_hr = os.path.join(NPY_RESULTS_DIR, f'HR_{tag}.npy')
            
            try:
                # Load the numpy arrays from the files.
                img_lr = np.load(path_lr)
                img_sr = np.load(path_sr)
                img_hr = np.load(path_hr)
                
                # Create a figure to hold the three power spectrum plots side-by-side.
                plt.figure(figsize=(18, 5))
                
                # Plot the spectrum for the Low-Resolution input.
                plt.subplot(1, 3, 1)
                plot_power_spectrum(img_lr, f'LR Power Spectrum ({tag})')
                
                # Plot the spectrum for our model's Super-Resolution output.
                plt.subplot(1, 3, 2)
                plot_power_spectrum(img_sr, f'SR Power Spectrum ({tag})')
                
                # Plot the spectrum for the High-Resolution ground truth.
                plt.subplot(1, 3, 3)
                plot_power_spectrum(img_hr, f'HR Power Spectrum ({tag})')
                
                # Save the final comparison image.
                output_filename = f'power_spectrum_{tag}.png'
                plt.suptitle('Frequency Domain Analysis', fontsize=16)
                plt.tight_layout(rect=[0, 0, 1, 0.96])
                plt.savefig(output_filename)
                plt.close()
                
                print(f"-> Saved analysis to {output_filename}")
                
            except FileNotFoundError as e:
                print(f"-> Skipping {tag}: Could not find a file. Error: {e}")
            except Exception as e:
                print(f"-> Skipping {tag}: An unexpected error occurred: {e}")

In [ ]:
# --- Load the first sample ---
tag = SAMPLE_TAGS[0]
sr_path = os.path.join(NPY_RESULTS_DIR, f"SR_{tag}.npy")
hr_path = os.path.join(NPY_RESULTS_DIR, f"HR_{tag}.npy")

# Load the numpy arrays (they are saved as [1, 1, H, W])
sr_np = np.load(sr_path)[0, 0]
hr_np = np.load(hr_path)[0, 0]

# --- 1. Visual Comparison ---
plt.figure(figsize=(18, 5))
plt.subplot(1, 3, 1)
plt.imshow(sr_np, cmap='gray')
plt.title(f'Super-Resolved (SR) - {tag}')
plt.axis('off')

plt.subplot(1, 3, 2)
plt.imshow(hr_np, cmap='gray')
plt.title(f'Ground Truth (HR) - {tag}')
plt.axis('off')

plt.subplot(1, 3, 3)
# Calculate and display the absolute difference
abs_diff = np.abs(sr_np - hr_np)
plt.imshow(abs_diff, cmap='hot')
plt.title('|SR - HR| Difference')
plt.colorbar()
plt.axis('off')

plt.tight_layout()
plt.show()

# --- 2. Frequency (Power Spectrum) Comparison ---
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plot_power_spectrum(sr_np, f'Power Spectrum of SR - {tag}')

plt.subplot(1, 2, 2)
plot_power_spectrum(hr_np, f'Power Spectrum of HR - {tag}')

plt.tight_layout()
plt.show()

In [ ]:
# --- 3. Line Profile Analysis ---
# Choose a row (y-coordinate) that crosses a sharp edge in the dose map
line_row = sr_np.shape[0] // 2 

plt.figure(figsize=(12, 6))
plt.plot(hr_np[line_row, :], label='Ground Truth (HR)', color='black', linewidth=2)
plt.plot(sr_np[line_row, :], label='Super-Resolved (SR)', color='red', linestyle='--')
plt.title(f'Intensity Profile across row {line_row}')
plt.xlabel('Pixel Column (x-coordinate)')
plt.ylabel('Normalized Intensity')
plt.grid(True, linestyle=':')
plt.legend()
plt.show()